# Grounded Legal Document Drafting System

In [1]:
!pip install transformers pymupdf pytesseract pdf2image pillow langchain-text-splitters sentence-transformers faiss-cpu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.0/25.0 MB 57.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 63.5 MB/s eta 0:00:00


In [4]:
import fitz
import pytesseract
import io
import os

from huggingface_hub import login
from PIL import Image, ImageFilter, ImageOps


d:\Assignment Project - Legal AI System\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:

# # Retrieve the token from Colab secrets
# hf_token = os.environ.get('HF_TOKEN')

# # Log in to Hugging Face
# login(token=hf_token)

# print("Successfully logged in to Hugging Face.")

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


Successfully logged in to Hugging Face.


### Initially tried to use Gemma3-1B : Bad Results, Gemma3-4B: Took too long to run

In [ ]:
# from transformers import pipeline
# import torch

# pipe = pipeline("text-generation", model="google/gemma-3-4b-it", device="cpu")

# messages = [
#     [
#         {
#             "role": "system",
#             "content": [{"type": "text", "text": "You are a helpful assistant."},]
#         },
#         {
#             "role": "user",
#             "content": [{"type": "text", "text": "Write a poem on Hugging Face, the company"},]
#         },
#     ],
# ]

# output = pipe(messages, max_new_tokens=50)

In [ ]:
# output

[[{'generated_text': [{'role': 'system',
     'content': [{'type': 'text', 'text': 'You are a helpful assistant.'}]},
    {'role': 'user',
     'content': [{'type': 'text',
       'text': 'Write a poem on Hugging Face, the company'}]},
    {'role': 'assistant',
     'content': "Okay, here's a poem about Hugging Face, aiming for a blend of admiration and a touch of technical wonder:\n\n**The Neural Bloom**\n\nWithin the cloud, a vibrant hue,\nHugging Face, a digital view.\n"}]}]]

### GROQ Test

In [5]:
import requests
import os
import json

groq_api_key = os.environ.get("GROQ_KEY")
url = "https://api.groq.com/openai/v1/models"

headers = {
    "Authorization": f"Bearer {groq_api_key}",
    "Content-Type": "application/json"
}

response = requests.get(url, headers=headers)

# Check if the request was successful
response.raise_for_status()

# Get the JSON content from the response
pretty_json = json.dumps(response.json(), indent=4)
print(pretty_json)

{
    "object": "list",
    "data": [
        {
            "id": "llama-3.3-70b-versatile",
            "object": "model",
            "created": 1733447754,
            "owned_by": "Meta",
            "active": true,
            "context_window": 131072,
            "public_apps": null,
            "max_completion_tokens": 32768
        },
        {
            "id": "whisper-large-v3",
            "object": "model",
            "created": 1693721698,
            "owned_by": "OpenAI",
            "active": true,
            "context_window": 448,
            "public_apps": null,
            "max_completion_tokens": 448
        },
        {
            "id": "meta-llama/llama-prompt-guard-2-22m",
            "object": "model",
            "created": 1748632101,
            "owned_by": "Meta",
            "active": true,
            "context_window": 512,
            "public_apps": null,
            "max_completion_tokens": 512
        },
        {
            "id": "llama-3.1-8b-insta

In [6]:
from groq import Groq

client = Groq(api_key=groq_api_key)

response = client.chat.completions.create(
    model="llama-3.1-8b-instant",
    messages=[
        {
            "role": "user",
            "content": "Give me a dad joke"
        }
    ]
)

print(response.choices[0].message.content)

Here's one:

Why did the scarecrow win an award?

Because he was outstanding in his field.


### PDF Loading

In [7]:
doc = fitz.open("../pdf_samples/sample_legal_case_packet.pdf")

text = ""

for page in doc:
    text += page.get_text()

print(text)

Case File: Harper vs. Westbrook Holdings
This packet contains partially structured legal-style documents intended for OCR, retrieval, grounded
summarization, and evidence extraction experiments. Some sections intentionally contain inconsistent
formatting and noisy data.
Case Summary
Plaintiff: Amelia Harper
Defendant: Westbrook Holdings LLC
Case Type: Property Ownership Dispute
Relevant Dates: March 12, 2022 — Property transfer allegedly recorded. June 4, 2023 — Inspection
request submitted. January 11, 2024 — Ownership challenge filed. The claimant alleges the transfer
records contain conflicting ownership information.
Document
Page
Evidence Snippet
Transfer Record
3
Ownership remains under review
Inspection Memo
5
Boundary markers unclear
Witness Statement
7
Prior owner disputed transaction
SCANNED NOTE (low quality transcription): “Owner ship certifcate appears incomplete. signture
mismatch on record copy. Need manual verificatoin before filing recommendation.”
Attached Field Note S

In [8]:
PDF_PATH = "../pdf_samples/sample_legal_case_packet.pdf"

doc = fitz.open(PDF_PATH)

all_pages = []

for page_num, page in enumerate(doc, start=1):

    # -----------------------------------
    # Native text extraction
    # -----------------------------------
    native_text = page.get_text().strip()

    # -----------------------------------
    # Find embedded images
    # -----------------------------------
    image_list = page.get_images(full=True)

    ocr_texts = []
    used_ocr_for_page = False  # Initialize flag for current page

    for img_index, img in enumerate(image_list):

        xref = img[0]

        # Extract image bytes
        base_image = doc.extract_image(xref)

        image_bytes = base_image["image"]

        image = Image.open(io.BytesIO(image_bytes))

        # -----------------------------------
        # Preprocessing
        # -----------------------------------
        image = ImageOps.grayscale(image)

        image = ImageOps.autocontrast(image)

        image = image.filter(ImageFilter.SHARPEN)

        # -----------------------------------
        # OCR image ONLY
        # -----------------------------------
        text = pytesseract.image_to_string(image)

        if text.strip():
            ocr_texts.append(text)
            used_ocr_for_page = True  # Set flag if OCR text is found

    # -----------------------------------
    # Merge native + OCR image text
    # -----------------------------------
    final_text = native_text

    if ocr_texts:

        final_text += "\n\n[OCR IMAGE CONTENT]\n"

        final_text += "\n".join(ocr_texts)

    all_pages.append({
        "page": page_num,
        "num_images": len(image_list),
        "text": final_text,
        "used_ocr": used_ocr_for_page  # Add the new key
    })

# -----------------------------------
# Preview
# -----------------------------------

for page_data in all_pages:

    print("=" * 80)
    print(f"PAGE: {page_data['page']}")
    print(f"IMAGES FOUND: {page_data['num_images']}")
    print("=" * 80)

    print(page_data["text"][:2000])
    print("\n")

PAGE: 1
IMAGES FOUND: 1
Case File: Harper vs. Westbrook Holdings
This packet contains partially structured legal-style documents intended for OCR, retrieval, grounded
summarization, and evidence extraction experiments. Some sections intentionally contain inconsistent
formatting and noisy data.
Case Summary
Plaintiff: Amelia Harper
Defendant: Westbrook Holdings LLC
Case Type: Property Ownership Dispute
Relevant Dates: March 12, 2022 — Property transfer allegedly recorded. June 4, 2023 — Inspection
request submitted. January 11, 2024 — Ownership challenge filed. The claimant alleges the transfer
records contain conflicting ownership information.
Document
Page
Evidence Snippet
Transfer Record
3
Ownership remains under review
Inspection Memo
5
Boundary markers unclear
Witness Statement
7
Prior owner disputed transaction
SCANNED NOTE (low quality transcription): “Owner ship certifcate appears incomplete. signture
mismatch on record copy. Need manual verificatoin before filing recommendation

In [9]:
documents = []

for page_data in all_pages:

    documents.append({
        "text": page_data["text"],
        "metadata": {
            "page": page_data["page"],
            "source": PDF_PATH
        }
    })
documents

[{'text': 'Case File: Harper vs. Westbrook Holdings\nThis packet contains partially structured legal-style documents intended for OCR, retrieval, grounded\nsummarization, and evidence extraction experiments. Some sections intentionally contain inconsistent\nformatting and noisy data.\nCase Summary\nPlaintiff: Amelia Harper\nDefendant: Westbrook Holdings LLC\nCase Type: Property Ownership Dispute\nRelevant Dates: March 12, 2022 — Property transfer allegedly recorded. June 4, 2023 — Inspection\nrequest submitted. January 11, 2024 — Ownership challenge filed. The claimant alleges the transfer\nrecords contain conflicting ownership information.\nDocument\nPage\nEvidence Snippet\nTransfer Record\n3\nOwnership remains under review\nInspection Memo\n5\nBoundary markers unclear\nWitness Statement\n7\nPrior owner disputed transaction\nSCANNED NOTE (low quality transcription): “Owner ship certifcate appears incomplete. signture\nmismatch on record copy. Need manual verificatoin before filing rec

##Split Document

In [10]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=200,
    separators=["\n\n", "\n", "."," "]

)


In [11]:
chunks = []

for doc in documents:

    split_texts = splitter.split_text(doc["text"])
    print(split_texts)

    for chunk in split_texts:

        chunks.append({
            "text": chunk,
            "metadata": doc["metadata"]
        })

['Case File: Harper vs. Westbrook Holdings\nThis packet contains partially structured legal-style documents intended for OCR, retrieval, grounded\nsummarization, and evidence extraction experiments. Some sections intentionally contain inconsistent\nformatting and noisy data.\nCase Summary\nPlaintiff: Amelia Harper\nDefendant: Westbrook Holdings LLC\nCase Type: Property Ownership Dispute\nRelevant Dates: March 12, 2022 — Property transfer allegedly recorded. June 4, 2023 — Inspection', 'Plaintiff: Amelia Harper\nDefendant: Westbrook Holdings LLC\nCase Type: Property Ownership Dispute\nRelevant Dates: March 12, 2022 — Property transfer allegedly recorded. June 4, 2023 — Inspection\nrequest submitted. January 11, 2024 — Ownership challenge filed. The claimant alleges the transfer\nrecords contain conflicting ownership information.\nDocument\nPage\nEvidence Snippet\nTransfer Record\n3\nOwnership remains under review\nInspection Memo\n5\nBoundary markers unclear\nWitness Statement\n7', 'rec

In [12]:
chunks

[{'text': 'Case File: Harper vs. Westbrook Holdings\nThis packet contains partially structured legal-style documents intended for OCR, retrieval, grounded\nsummarization, and evidence extraction experiments. Some sections intentionally contain inconsistent\nformatting and noisy data.\nCase Summary\nPlaintiff: Amelia Harper\nDefendant: Westbrook Holdings LLC\nCase Type: Property Ownership Dispute\nRelevant Dates: March 12, 2022 — Property transfer allegedly recorded. June 4, 2023 — Inspection',
  'metadata': {'page': 1,
   'source': '../pdf_samples/sample_legal_case_packet.pdf'}},
 {'text': 'Plaintiff: Amelia Harper\nDefendant: Westbrook Holdings LLC\nCase Type: Property Ownership Dispute\nRelevant Dates: March 12, 2022 — Property transfer allegedly recorded. June 4, 2023 — Inspection\nrequest submitted. January 11, 2024 — Ownership challenge filed. The claimant alleges the transfer\nrecords contain conflicting ownership information.\nDocument\nPage\nEvidence Snippet\nTransfer Record\n3

## Create Embeddings

In [14]:
from sentence_transformers import SentenceTransformer

embedding_model = SentenceTransformer(
    "BAAI/bge-small-en-v1.5"
)

texts = [chunk["text"] for chunk in chunks]

embeddings = embedding_model.encode(
    texts,
    show_progress_bar=True
)

Batches: 100%|██████████| 1/1 [00:00<00:00,  3.62it/s]


### FAISS Vector DB

In [15]:
import faiss
import numpy as np

embedding_dim = embeddings.shape[1]

index = faiss.IndexFlatL2(embedding_dim)

index.add(np.array(embeddings).astype("float32"))

In [16]:
query = "Summarize ownership dispute evidence"
query_embedding = embedding_model.encode([query])
D, I = index.search(
    np.array(query_embedding).astype("float32"),
    k=5
)
retrieved_chunks = [chunks[i] for i in I[0]]


In [17]:
context = ""

for i, chunk in enumerate(retrieved_chunks):

    context += f"""
    [Evidence {i+1}]
    Source Page: {chunk['metadata']['page']}

    {chunk['text']}
    """

### Structured Draft Generation

In [18]:
prompt = f"""
You are generating a grounded legal-style summary.

ONLY use the provided evidence.

If information is uncertain,
explicitly say uncertain.

Retrieved Evidence:

{context}

Generate:
1. Case summary
2. Key timeline
3. Important unresolved issues
4. Evidence references
"""

In [19]:
from groq import Groq

client = Groq(api_key=groq_api_key)

response = client.chat.completions.create(
    model="llama-3.1-8b-instant",
    messages=[
        {
            "role": "system",
            "content": "You are a helpful assistant."
        },
        {
            "role": "user",
            "content": prompt
        },
    ]
)

print(response.choices[0].message.content)

**Case Summary**
The case at hand is Harper vs. Westbrook Holdings, a property ownership dispute initiated by Plaintiff Amelia Harper against Defendant Westbrook Holdings LLC. The dispute revolves around conflicting ownership information in the transfer records of a property. Key aspects of the dispute include incomplete or potentially false documentation, unclear boundary markers, and a need for manual verification to determine the true ownership of the property.

**Key Timeline**

- March 12, 2022: The property allegedly transferred with conflicting ownership information reportedly recorded.
- June 4, 2023: The plaintiff submitted an inspection request to investigate the disputed ownership.
- January 11, 2024: The plaintiff filed the ownership challenge.

**Important Unresolved Issues**

1. Uncertainty about the authenticity of the transfer records due to the alleged ownership remains under review.
2. Boundary markers are unclear, making it uncertain what the actual property boundari

## Feedback Loop

In [20]:
import json
import uuid
from datetime import datetime
from pathlib import Path

EDITS_FILE = Path("edit_records.json")
RULES_FILE = Path("learned_rules.json")

# -----------------------------------
# Persistence helpers
# -----------------------------------

def load_json(path):
    return json.loads(path.read_text()) if path.exists() else []

def save_json(path, data):
    path.write_text(json.dumps(data, indent=2))

# -----------------------------------
# Capture an operator edit
# -----------------------------------

def capture_edit(original, edited, doc_id, query):
    record = {
        "id": str(uuid.uuid4()),
        "timestamp": datetime.utcnow().isoformat(),
        "doc_id": doc_id,
        "query": query,
        "original_draft": original,
        "edited_draft": edited,
        "rule_extracted": False,
    }
    edits = load_json(EDITS_FILE)
    edits.append(record)
    save_json(EDITS_FILE, edits)
    print(f"Captured edit: {record['id']}")
    return record

# -----------------------------------
# Extract a rule from one edit via LLM
# -----------------------------------

EXTRACT_SYSTEM = """You are an expert legal writing analyst.
Study how an operator corrected an AI draft and return ONE concise reusable
drafting instruction (1-3 sentences, imperative form, no preamble, no explanation)."""

EXTRACT_USER = """Original draft:
{original}

Edited version:
{edited}

What single reusable drafting rule does this edit teach?"""

def extract_rule(edit):
    response = client.chat.completions.create(
        model="llama-3.1-8b-instant",
        messages=[
            {"role": "system", "content": EXTRACT_SYSTEM},
            {"role": "user",   "content": EXTRACT_USER.format(
                original=edit["original_draft"],
                edited=edit["edited_draft"]
            )},
        ],
        temperature=0.2,
        max_tokens=200,
    )
    return response.choices[0].message.content.strip()

def process_pending_edits():
    edits = load_json(EDITS_FILE)
    rules = load_json(RULES_FILE)
    new_rules = []

    for edit in edits:
        if edit["rule_extracted"]:
            continue

        print(f"Extracting rule from edit {edit['id']} ...")
        rule_text = extract_rule(edit)
        rule = {
            "rule": rule_text,
            "source_edit_id": edit["id"],
            "source_doc_id": edit["doc_id"],
        }
        rules.append(rule)
        new_rules.append(rule)
        edit["rule_extracted"] = True
        print(f"Rule: {rule_text}")

    save_json(EDITS_FILE, edits)
    save_json(RULES_FILE, rules)
    return new_rules

# -----------------------------------
# Build rules block for prompt injection
# -----------------------------------

def build_rules_block():
    rules = load_json(RULES_FILE)
    if not rules:
        return ""
    lines = ["Learned operator preferences — apply all of these:\n"]
    for i, r in enumerate(rules[-10:][::-1], 1):
        lines.append(f"  {i}. {r['rule']}")
    return "\n".join(lines)

print("Feedback helpers loaded.")

Feedback helpers loaded.


In [21]:
# These represent realistic corrections a legal operator would make.
# In production these come from a UI where the operator edits the draft directly.

simulated_edits = [
    {
        "original": (
            "The ownership certificate appears incomplete with a signature mismatch "
            "on the record copy, requiring manual verification before filing."
        ),
        "edited": (
            "NOTE: The following is derived from a low-quality OCR scan and should "
            "NOT be treated as verified evidence until manually reviewed. "
            "The scan suggests the ownership certificate may be incomplete and "
            "a signature mismatch may exist on the record copy."
        ),
    },
    {
        "original": (
            "**Key Timeline**\n"
            "1. March 12, 2022: Property transfer allegedly recorded\n"
            "2. June 4, 2023: Inspection request submitted\n"
            "3. January 11, 2024: Ownership challenge filed\n\n"
            "**Important Unresolved Issues**\n"
            "- Signature mismatch on ownership certificate\n"
            "- Unclear boundary markers"
        ),
        "edited": (
            "**Important Unresolved Issues** *(listed first for reviewer attention)*\n"
            "- Signature mismatch on ownership certificate (unverified — OCR source)\n"
            "- Unclear boundary markers (Inspection Memo, Page 5)\n"
            "- OCR content from scanned note requires manual verification\n\n"
            "**Key Timeline**\n"
            "1. March 12, 2022: Property transfer allegedly recorded\n"
            "2. June 4, 2023: Inspection request submitted\n"
            "3. January 11, 2024: Ownership challenge filed"
        ),
    },
]

for edit in simulated_edits:
    capture_edit(
        original=edit["original"],
        edited=edit["edited"],
        doc_id="harper_vs_westbrook",
        query=query,
    )

Captured edit: 2e7e5a30-e839-4ea6-afda-034e08724550
Captured edit: bd707dc7-80b7-416a-9a1f-858366dd647d


In [22]:
new_rules = process_pending_edits()

print("\nAll learned rules:")
for r in load_json(RULES_FILE):
    print(f"  • {r['rule']}")

Extracting rule from edit 2e7e5a30-e839-4ea6-afda-034e08724550 ...
Rule: Use cautionary language when referencing unverified or potentially unreliable sources.
Extracting rule from edit bd707dc7-80b7-416a-9a1f-858366dd647d ...
Rule: Clarify the source and reliability of information in parentheses, especially when referencing external or automated sources.

All learned rules:
  • Use cautionary language when referencing unverified or potentially unreliable sources.
  • Clarify the source and reliability of information in parentheses, especially when referencing external or automated sources.


In [23]:
baseline_prompt = f"""
You are generating a grounded legal-style summary.

ONLY use the provided evidence.

If information is uncertain,
explicitly say uncertain.

Retrieved Evidence:

{context}

Generate:
1. Case summary
2. Key timeline
3. Important unresolved issues
4. Evidence references
"""

baseline_response = client.chat.completions.create(
    model="llama-3.1-8b-instant",
    messages=[
        {
            "role": "system",
            "content": "You are a helpful assistant."
        },
        {
            "role": "user",
            "content": baseline_prompt
        },
    ]
)

baseline_draft = baseline_response.choices[0].message.content
print(baseline_draft)

**Case Summary**

Amelia Harper (Plaintiff) has filed a property ownership dispute case against Westbrook Holdings LLC (Defendant). The case revolves around the allegedly recorded property transfer on March 12, 2022, which contains conflicting ownership information. The inspection request was submitted on June 4, 2023, and the ownership challenge was filed on January 11, 2024.

**Key Timeline**

- March 12, 2022: Property transfer allegedly recorded
- June 4, 2023: Inspection request submitted
- Uncertain (after January 11, 2024, since exact date not specified): Manual verification of ownership certificate
- January 11, 2024: Ownership challenge filed

**Important Unresolved Issues**

1. The transfer records contain conflicting ownership information which needs investigation.
2. The boundary markers are unclear, as stated in the Inspection Memo.
3. There is a prior owner who disputed the transaction; exact details are uncertain.
4. The ownership certificate appears to be incomplete wit

In [24]:
rules_block = build_rules_block()

improved_prompt = f"""
You are generating a grounded legal-style summary.

{rules_block}

ONLY use the provided evidence.

If information is uncertain,
explicitly say uncertain.

Retrieved Evidence:

{context}

Generate:
1. Case summary
2. Key timeline
3. Important unresolved issues
4. Evidence references
"""

improved_response = client.chat.completions.create(
    model="llama-3.1-8b-instant",
    messages=[
        {
            "role": "system",
            "content": "You are a helpful assistant."
        },
        {
            "role": "user",
            "content": improved_prompt
        },
    ]
)

improved_draft = improved_response.choices[0].message.content
print(improved_draft)

**Case Summary**

This is a property ownership dispute between Plaintiff Amelia Harper and Defendant Westbrook Holdings LLC. The dispute centers on allegedly conflicting ownership information contained in transfer records.

Sources: 
(Transfer Record, Source Page: 1, 
Inspection Memo, Source Page: 1, and 
Witness Statement, Source Page: 1 (retrieved from the same source))


**Key Timeline**

1. March 12, 2022: Property transfer allegedly recorded.
2. June 4, 2023: Inspection request submitted.
 (Uncertain date) : Prior owner disputed transaction.
3. January 11, 2024: Ownership challenge filed by Plaintiff.
 

Sources:
[Not specified due to unclear nature of 'prior owner disputed transaction']. 
Relevant dates found within evidence: 
(Transfer Record, Source Page: 1, 
Inspection Memo, Source Page: 1, and 
Case Summary, Evidence 3)

**Important Unresolved Issues**

The key unresolved issues in this case include:

1. Conflicting ownership information in transfer records.
2. Unclear bounda

In [25]:
print("=" * 80)
print("RULES APPLIED")
print("=" * 80)
print(rules_block)

print("\n" + "=" * 80)
print("BASELINE DRAFT (no learned rules)")
print("=" * 80)
print(baseline_draft)

print("\n" + "=" * 80)
print("IMPROVED DRAFT (after operator feedback)")
print("=" * 80)
print(improved_draft)

RULES APPLIED
Learned operator preferences — apply all of these:

  1. Clarify the source and reliability of information in parentheses, especially when referencing external or automated sources.
  2. Use cautionary language when referencing unverified or potentially unreliable sources.

BASELINE DRAFT (no learned rules)
**Case Summary**

Amelia Harper (Plaintiff) has filed a property ownership dispute case against Westbrook Holdings LLC (Defendant). The case revolves around the allegedly recorded property transfer on March 12, 2022, which contains conflicting ownership information. The inspection request was submitted on June 4, 2023, and the ownership challenge was filed on January 11, 2024.

**Key Timeline**

- March 12, 2022: Property transfer allegedly recorded
- June 4, 2023: Inspection request submitted
- Uncertain (after January 11, 2024, since exact date not specified): Manual verification of ownership certificate
- January 11, 2024: Ownership challenge filed

**Important Unre